### TOOLS

Models can request to call tools that perform task such as fetching data from a database,searching the web,or running code. tools are pairings of:  
   
1.A schema including the name of the tool,a description,and/or argument definitions (often a JSON schema)

2.A function or coroutine to execute.

In [ ]:
### TOOLS EXPLANATION (your text is fine)

# -------------------------------
# STEP 1: LOAD ENV + MODEL
# -------------------------------
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# Debug (optional)
print(os.getenv("GROQ_API_KEY"))

# Create model
model = init_chat_model("groq:llama-3.1-8b-instant")


# -------------------------------
# STEP 2: NORMAL MODEL CALL
# -------------------------------
response = model.invoke("write me an Essay on AI")
print(response.content)


# -------------------------------
# STEP 3: DEFINE TOOL
# -------------------------------
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


# Bind tool to model
model_with_tools = model.bind_tools([get_weather])


# -------------------------------
# STEP 4: TEST TOOL CALL (IMPORTANT FIX)
# -------------------------------
from langchain_core.messages import HumanMessage

response = model_with_tools.invoke(
    [HumanMessage(content="what's the weather in Hyderabad")]
)

print(response)
print(response.tool_calls)   # see tool call


# -------------------------------
# STEP 5: TOOL EXECUTION LOOP
# -------------------------------

# Step 1: Model generates tool call
messages = [HumanMessage(content="what's the weather in Hyderabad?")]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tool
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call["args"])   # ✅ FIXED
    messages.append(tool_result)

# Step 3: Final response
final_response = model_with_tools.invoke(messages)

print(final_response.content)